# Vectorized Operations

## Definition
**Vectorized operations** in pandas operate on entire columns/rows at once using underlying NumPy arrays
— no Python loops. They are the **fastest** way to transform data in pandas.

### Vectorized vs apply() vs loop
```
1M row DataFrame:
  Python loop:    ~5s
  .apply():       ~0.5s
  Vectorized:     ~0.005s  ← 1000x faster!
```

### Categories
| Type | Examples |
|------|---------|
| Arithmetic | `df['a'] + df['b']`, `df['x'] ** 2` |
| Comparison | `df['age'] > 30` |
| String | `df['name'].str.upper()` |
| DateTime | `df['date'].dt.year` |
| Numeric | `np.log(df['income'])`, `df['x'].abs()` |
| Aggregation | `df['score'].cumsum()`, `.diff()` |

In [ ]:
import pandas as pd
import numpy as np
import time

rng = np.random.default_rng(42)
N = 500_000
df = pd.DataFrame({
    'a': rng.integers(1, 100, N),
    'b': rng.integers(1, 100, N),
    'price': rng.uniform(10, 1000, N),
    'qty': rng.integers(1, 50, N),
    'discount': rng.uniform(0, 0.3, N),
    'name': ['item_' + str(i % 100) for i in range(N)]
})
print(df.shape)

In [ ]:
# Arithmetic vectorization
df['revenue'] = df['price'] * df['qty']
df['discounted_price'] = df['price'] * (1 - df['discount'])
df['profit'] = df['revenue'] * 0.2
df['a_plus_b'] = df['a'] + df['b']
df['ratio'] = df['a'] / (df['b'] + 1e-8)
print(df[['a', 'b', 'revenue', 'discounted_price']].head())

In [ ]:
# NumPy ufuncs work directly on DataFrame columns
df['log_price'] = np.log1p(df['price'])    # log(1 + price)
df['sqrt_qty'] = np.sqrt(df['qty'])
df['price_sq'] = np.square(df['price'])

# np.where — vectorized if-else
df['category'] = np.where(df['price'] > 500, 'expensive', 'affordable')
print(df[['price', 'log_price', 'sqrt_qty', 'category']].head())

In [ ]:
# Performance: vectorized vs apply
t0 = time.time()
result_apply = df['price'].apply(lambda x: np.log(x + 1))
t_apply = time.time() - t0

t0 = time.time()
result_vec = np.log1p(df['price'])
t_vec = time.time() - t0

print(f'apply():    {t_apply:.4f}s')
print(f'vectorized: {t_vec:.4f}s')
print(f'Speedup: {t_apply/t_vec:.0f}x')
print('Results match:', np.allclose(result_apply, result_vec))

In [ ]:
# Cumulative and diff operations
s = pd.Series([10, 20, 30, 40, 50])
print('cumsum:', s.cumsum().values)
print('cumprod:', s.cumprod().values)
print('diff():', s.diff().values)      # s[i] - s[i-1]
print('pct_change:', s.pct_change().round(3).values)

In [ ]:
# clip — vectorized clamping
df['price_clipped'] = df['price'].clip(lower=50, upper=900)
print('Before clip:', df['price'].describe()[['min', 'max']].to_dict())
print('After clip:', df['price_clipped'].describe()[['min', 'max']].to_dict())

In [ ]:
# abs, round, floor, ceil
s = pd.Series([-2.7, -1.5, 0.3, 1.5, 2.9])
print('abs:', s.abs().values)
print('round:', s.round(0).values)
print('floor:', np.floor(s).values)
print('ceil:', np.ceil(s).values)